<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">5. Lakehouse Federation</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 5.6 Lab Implementing Lakehouse Federation

In this lab you stand up a **Lakebase Autoscaling** Postgres database, seed it with a small operational table that enriches the TPC-DS customers, then federate it back into Unity Catalog as a foreign catalog. The cross-system join at the end combines Lakebase-resident customer enrichment with a narrow UC-resident slice of TPC-DS `store_sales` - same TPC-DS theme used throughout the course.

Lakebase is normally the *operational* side of a lakehouse architecture; here we are using it as a stand-in for an external Postgres instance to give you a hands-on Lakehouse Federation exercise. The federation mechanics (`CONNECTION` + `FOREIGN CATALOG`) are identical to what you would set up against any external Postgres instance.

## Learning Objectives

By the end of this lab, you will be able to:
- Stand up a Lakebase Autoscaling Postgres project and create a table
- Create a Unity Catalog `CONNECTION` to a Postgres instance using credentials
- Register a `FOREIGN CATALOG` over a Postgres database
- Run a standalone federated query and a cross-system join with a UC-native fact table
- Inspect query pushdown to confirm what work was delegated to Postgres

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li><strong>Serverless Compute, Version 5</strong>: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #2272B4">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
<p style="margin: 8px 0 0 0; color: #333">You must have run <strong>0 - Required Setup</strong> first. The Lakebase steps require <strong>Lakebase Autoscaling</strong> in your workspace region (see <a href="https://docs.databricks.com/aws/en/oltp/projects/manage-projects#availability" style="color: #2272B4">region availability</a>).</p>
</div>
</div>
</div>

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Lakebase Overview</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><strong>Lakebase</strong> is Databricks-managed PostgreSQL designed for OLTP workloads next to the lakehouse. <strong>Lakebase Autoscaling</strong> is the current generation, with autoscaling compute, scale-to-zero, branching, and instant restore.</li>
                <li>A <strong>project</strong> is the top-level Lakebase container. Each project has one or more <strong>branches</strong> (<code>production</code> is the default), and each branch holds a Postgres database (<code>databricks_postgres</code> by default).</li>
                <li>You connect with any standard Postgres client (<code>psql</code>, pgAdmin, DBeaver) using a Postgres connection string. Authentication is via Databricks OAuth - you paste an OAuth token as the password.</li>
                <li>Lakebase has its own SQL Editor in the workspace UI - that is what we use in this lab to create the table.</li>
            </ul>
            <p style="margin: 12px 0 0 0; color: #333;">Lakebase has deeper integrations with Unity Catalog (registering a Lakebase database as a UC catalog directly, synced tables, lakehouse sync via CDC) - we deliberately bypass those here. The point of this lab is to apply the <strong>federation</strong> pattern to a Postgres instance, treating Lakebase as if it were any external Postgres source.</p>
        </div>
    </div>
</div>

In [0]:
%run ../Includes/Classroom-Setup-5-lab

## A. Provision the Lakebase Database

Three steps in the Lakebase UI: create a project, capture the connection string, create and seed the table.

### A1. Create a Lakebase Autoscaling Project

From the workspace **apps switcher** (the icon grid in the top-right), open the **Lakebase** app. Select **Autoscaling** and click **New project**. Name the project using the __`lakebase_project_name`__ returned in the setup, accept the default Postgres version, and create. The project is created with a single `production` branch, a default `databricks_postgres` database, and compute that is always-on by default.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase:</span> What you should see (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">After creation:</p>
    <ul style="margin: 0 0 0 20px; color: #333;">
      <li>Your project listed under the Lakebase Autoscaling UI</li>
      <li>A <code>production</code> branch with status <em>Active</em></li>
      <li>A default database <code>databricks_postgres</code> ready to accept connections</li>
      <li>The region matches your workspace region</li>
    </ul>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase:</span> Apps Switcher Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-apps-switcher.png" alt="Workspace apps switcher with the Lakebase tile highlighted" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase:</span> Create Project Dialog Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-create-project-dialog.png" alt="Lakebase Autoscaling new project dialog showing project name and Postgres version" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A2. Capture the Connection Details

From your project, select the **production** branch and click **Connect**. In the Connect dialog, set the **Role** dropdown to your Databricks email (under OAuth roles). The dialog generates a `psql` connection string containing the values you need for the UC `CONNECTION` in Task B1.

Copy these values into a scratch space. You will paste them into Section B.

| What you need | Where to find it | Example |
|--------------|-----------------|---------|
| **host** | The hostname in the `psql` connection string (between `@` and the next `/`) | `ep-abc-123.staging.cloud.databricks.com` |
| **port** | Always `5432` for Lakebase | `5432` |
| **database** | The database name in the `psql` connection string (after the last `/`). Used later for the `FOREIGN CATALOG` in Section B, not the `CONNECTION`. | `databricks_postgres` |
| **user** | Your email address (shown in the Role dropdown) | `user@databricks.com` |
| **password** | Click **Copy OAuth token** at the bottom of the dialog. Use this token as the password. | (a long token string) |

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Lakebase';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase:</span> Connect Dialog Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-connect-oauth.png" alt="Lakebase Connect dialog showing the OAuth psql connection string for the production branch" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### A3. Create and Seed the `customer_enrichment` Table

Open the Lakebase **SQL Editor** for your `production` branch and run the SQL in the pulldown below. It creates a small `customer_enrichment` table keyed on `c_customer_sk` (the same surrogate key TPC-DS uses) and seeds 50 rows that overlap with the TPC-DS customer space.

In a real OLTP system this table would hold operational attributes - loyalty tier, marketing segment, last-touch channel, churn risk score - that change frequently and live in the transactional store. Joining it with the analytical fact table on the lakehouse side is the canonical Lakehouse Federation use case.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase:</span> Create + seed customer_enrichment (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
-- Run in the Lakebase SQL Editor against the production branch / databricks_postgres database.
<br/>
CREATE SCHEMA IF NOT EXISTS tpcds_ops;
<br/>
-- Drop and recreate so this block can be re-run safely. First run will
-- show "does not exist, skipping" — this is expected.
DROP TABLE IF EXISTS tpcds_ops.customer_enrichment;
<br/>
CREATE TABLE tpcds_ops.customer_enrichment (
    c_customer_sk      BIGINT       PRIMARY KEY,
    loyalty_tier       VARCHAR(16)  NOT NULL,
    marketing_segment  VARCHAR(32)  NOT NULL,
    churn_risk_score   NUMERIC(4,3) NOT NULL,
    last_touch_channel VARCHAR(16)  NOT NULL,
    updated_at         TIMESTAMPTZ  NOT NULL DEFAULT now()
);
<br/>
-- Seed 50 rows. c_customer_sk values 1..50 line up with the same range
-- in samples.tpcds_sf1000.customer, so the cross-system join in Section C
-- has matches on both sides.
INSERT INTO tpcds_ops.customer_enrichment
    (c_customer_sk, loyalty_tier, marketing_segment, churn_risk_score, last_touch_channel)
SELECT
    sk,
    CASE (sk % 4) WHEN 0 THEN 'platinum' WHEN 1 THEN 'gold' WHEN 2 THEN 'silver' ELSE 'bronze' END,
    CASE (sk % 5) WHEN 0 THEN 'value-seeker' WHEN 1 THEN 'fashion-forward'
                  WHEN 2 THEN 'family-buyer' WHEN 3 THEN 'tech-enthusiast' ELSE 'occasional' END,
    ROUND((random())::NUMERIC, 3),
    CASE (sk % 3) WHEN 0 THEN 'email' WHEN 1 THEN 'mobile-app' ELSE 'web' END
FROM generate_series(1, 50) AS s(sk);
<br/>
-- Smoke check
SELECT loyalty_tier, COUNT(*) AS customers
FROM tpcds_ops.customer_enrichment
GROUP BY loyalty_tier
ORDER BY loyalty_tier;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Lakebase';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Lakebase:</span> SQL Editor Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/lakebase-sql-create-table.png" alt="Lakebase SQL Editor preloaded with sample CREATE TABLE / INSERT statements" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

## B. Federate Lakebase from Unity Catalog

With the Lakebase table in place, the federation side is the standard two-object pattern: a `CONNECTION` (with credentials) and a `FOREIGN CATALOG` (the UC namespace surface). From here on, treat Lakebase as if it were any external Postgres instance.

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em;">Task B1 - Create the Lakebase Connection</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Create a UC <code>CONNECTION</code> of type <code>postgresql</code> to your Lakebase project. Replace the placeholders below with the values from A2:</p>
            <ul style="margin: 8px 0 0 16px; color: #333;">
              <li><code>&lt;lakebase-host&gt;</code> = the hostname from the Connect dialog (e.g. <code>ep-abc-123.staging.cloud.databricks.com</code>)</li>
              <li><code>&lt;your-email@databricks.com&gt;</code> = your Databricks email (the Role in the Connect dialog)</li>
              <li><code>&lt;oauth-token-or-pg-password&gt;</code> = the OAuth token copied from the Connect dialog (click <b>Copy OAuth token</b>)</li>
            </ul>
        </div>
    </div>
</div>

In [0]:
-- TODO: Create a connection to Lakebase Postgres
-- Hint: TYPE postgresql, OPTIONS host/port/password
-- For production, prefer secret('<scope>','<key>') over plaintext credentials.

  SET VAR pg_host_value = '{lakebase-host}';
  SET VAR pg_oauth_token_value = '{oauth-token}'; 
  SET VAR pg_create_connection_sql = CONCAT(
    'CREATE <FILL_IN> IF NOT EXISTS ', lab_pg_connection,
    ' <FILL_IN> postgresql OPTIONS (',
    'host ', q, pg_host_value, q, ', ',
    'port ', q, '5432', q, ', ',
    'user ', q, current_user(), q, ', ',
    'password ', q, pg_oauth_token_value, q,
    ')'
  );
  -- SELECT pg_create_connection_sql AS sql_to_run;
  EXECUTE IMMEDIATE pg_create_connection_sql;

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Show Solution</span> (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
SET VAR pg_host_value = '{lakebase-host}';
SET VAR pg_oauth_token_value = '{oauth-token}';
SET VAR pg_create_connection_sql = CONCAT(
   'CREATE CONNECTION IF NOT EXISTS ', lab_pg_connection,
   ' TYPE postgresql OPTIONS (',
   'host ', q, pg_host_value, q, ', ',
   'port ', q, '5432', q, ', ',
   'user ', q, current_user(), q, ', ',
   'password ', q, pg_oauth_token_value, q,
   ')'
 );


 -- SELECT pg_create_connection_sql AS sql_to_run;
 EXECUTE IMMEDIATE pg_create_connection_sql;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Databricks';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

In [0]:
-- Verify the connection landed.
EXECUTE IMMEDIATE CONCAT('DESCRIBE CONNECTION EXTENDED ', lab_pg_connection);

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em;">Task B2 - Create the Foreign Catalog</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Register a foreign catalog using the <code>lab_pg_catalog</code> name returned by the setup (per-user; see the setup output for your actual catalog name), over the Lakebase database <code>databricks_postgres</code>.</p>
        </div>
    </div>
</div>


In [0]:
-- TODO: Create the foreign catalog over the Lakebase database.

SET VAR pg_create_catalog_sql = CONCAT(
  'CREATE <FILL_IN> IF NOT EXISTS ', lab_pg_catalog,
  ' USING <FILL_IN> ', lab_pg_connection,
  ' OPTIONS (<FILL_IN> ', q, 'databricks_postgres', q, ')'
);
EXECUTE IMMEDIATE pg_create_catalog_sql;


<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Show Solution</span> (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
SET VAR pg_create_catalog_sql = CONCAT(
 'CREATE FOREIGN CATALOG IF NOT EXISTS ', lab_pg_catalog,
 ' USING CONNECTION ', lab_pg_connection,
 ' OPTIONS (database ', q, 'databricks_postgres', q, ')'
);


EXECUTE IMMEDIATE pg_create_catalog_sql;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Databricks';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

In [0]:
-- Confirm the catalog and the tpcds_ops schema appear.
EXECUTE IMMEDIATE CONCAT('SHOW SCHEMAS IN ', lab_pg_catalog);


## C. Federated Queries

Two queries: a standalone aggregate against the foreign catalog, then the headline cross-system join with the narrow `lab_store_sales_slice` (built at setup time so this lab does not scan the full TPC-DS fact).

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em;">Task C1 - Aggregate Customers by Loyalty Tier</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Count customer-enrichment rows grouped by <code>loyalty_tier</code>, ordered by count descending. This query reads exclusively from Lakebase - the aggregation is pushed down to Postgres.</p>
        </div>
    </div>
</div>

In [0]:
-- TODO: Aggregate the foreign customer_enrichment table
SELECT
  <FILL_IN> AS loyalty_tier,
  COUNT(*)  AS customers
FROM IDENTIFIER(lab_pg_catalog || '.tpcds_ops.customer_enrichment')
GROUP BY <FILL_IN>
ORDER BY customers DESC;


<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Show Solution</span> (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
SELECT
  loyalty_tier,
  COUNT(*) AS customers
FROM IDENTIFIER(lab_pg_catalog || '.tpcds_ops.customer_enrichment')
GROUP BY loyalty_tier
ORDER BY customers DESC;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Databricks';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## D. Cross-System Join - Lakebase + UC

The headline use case. Federated `customer_enrichment` from Lakebase joined with the UC-resident `lab_store_sales_slice` (a narrow, pre-filtered TPC-DS slice that `Classroom-Setup-5-lab` materialised at setup time). UC governs both halves of the join: the same grants, audit, and lineage apply across systems.

<div style="font-size: 1em; border-left: 4px solid #ffc107; background: #fffde7; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #ff8f00; font-size: 1.1em;">Task D1 - Net Revenue by Loyalty Tier</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Aggregate net revenue from <code>lab_store_sales_slice</code> by <code>loyalty_tier</code> from the federated <code>customer_enrichment</code> table. The slice is already filtered to a 30-day window at setup time, so no <code>WHERE</code> clause is required.</p>
        </div>
    </div>
</div>

In [0]:
-- TODO: Cross-system join - foreign customer_enrichment + UC fact slice.
-- The slice is already filtered to ss_sold_date_sk BETWEEN 2451180 AND 2451210
-- at setup time, so no WHERE clause is needed here.
SELECT
  ce.<FILL_IN>                  AS loyalty_tier,
  COUNT(*)                      AS line_items,
  ROUND(SUM(s.ss_net_paid), 2)  AS net_revenue
FROM data_interoperability_tpcds.lab_store_sales_slice s
JOIN IDENTIFIER(lab_pg_catalog || '.tpcds_ops.customer_enrichment') ce
  ON ce.c_customer_sk = s.ss_customer_sk
GROUP BY ce.<FILL_IN>
ORDER BY net_revenue DESC;


<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Show Solution</span> (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="sql">
SELECT
  ce.loyalty_tier,
  COUNT(*)                     AS line_items,
  ROUND(SUM(s.ss_net_paid), 2) AS net_revenue
FROM data_interoperability_tpcds.lab_store_sales_slice s
JOIN IDENTIFIER(lab_pg_catalog || '.tpcds_ops.customer_enrichment') ce
  ON ce.c_customer_sk = s.ss_customer_sk
GROUP BY ce.loyalty_tier
ORDER BY net_revenue DESC;
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var label = lang === 'bash' ? 'Terminal' : 'Databricks';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## E. Inspect Pushdown

`EXPLAIN FORMATTED` reveals which operations the planner pushed down to Postgres and which stayed in UC's compute layer. For Postgres sources, filters, projections, limits, and aggregates push down on Databricks Runtime 13.3+ / SQL warehouses 2023.40+.

In [0]:
EXPLAIN FORMATTED
SELECT loyalty_tier, COUNT(*)
FROM IDENTIFIER(lab_pg_catalog || '.tpcds_ops.customer_enrichment')
WHERE loyalty_tier IN ('platinum', 'gold')
GROUP BY loyalty_tier;


In [0]:
-- Remove the foreign catalog created in this lab.
EXECUTE IMMEDIATE CONCAT('DROP CATALOG IF EXISTS ', lab_pg_catalog);

-- Remove the PostgreSQL connection created in this lab.
EXECUTE IMMEDIATE CONCAT(
  'DROP CONNECTION IF EXISTS ',
  lab_pg_connection
);


## Lab Complete

This lab covered the full Lakehouse Federation workflow against Lakebase Postgres: provisioning the database, creating a UC CONNECTION and FOREIGN CATALOG, running federated queries and a cross-system join, and confirming pushdown with EXPLAIN FORMATTED.

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What You Built</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li>A <strong>Lakebase Autoscaling project</strong> with a seeded <code>customer_enrichment</code> table - operational data living in OLTP-shaped Postgres</li>
                <li>A UC <strong><code>CONNECTION</code></strong> and <strong><code>FOREIGN CATALOG</code></strong> over Postgres - the same pattern as 5.2's SQL Server / Snowflake demos</li>
                <li>A <strong>standalone federated query</strong> that pushed aggregation down to Postgres</li>
                <li>A <strong>cross-system join</strong> combining Lakebase-resident enrichment with the UC-resident TPC-DS fact table - operational + analytical data unified by UC governance</li>
                <li>An <strong><code>EXPLAIN FORMATTED</code></strong> walk showing which operations crossed the wire</li>
            </ul>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>